In [ ]:
!git clone https://github.com/Mikayla-ds2/surge-sense.git

In [ ]:
# dependecies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/cleaned-data.csv')

data = data.drop(['Unnamed: 0'], axis=1)

data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.shape

In [ ]:
import seaborn as sns
from matplotlib.colors import ListedColormap

In [ ]:
# for all my plots
palette = ['#EAD3A9', '#84592B', '#A05135',
           '#743015', '#462D1B', '#9D9368']
customcmap = ListedColormap(palette)

In [ ]:
# creating cross table to evaluate if admission is a valuable variable
crosstab01 = pd.crosstab(data['admission_flag'],
                         data['patient_waittime'])

plt.figure(figsize=(12, 7))
crosstab01.plot(kind='bar', stacked=True, colormap=customcmap)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
# determined by the previous images, admission_flag
# has inconclusive context and
# thus need to be excluded from any further analysis or modeling.
data = data.drop(['admission_flag'], axis=1)
data.head()

In [ ]:
histogram = data.select_dtypes(
    include=['number', 'bool']).hist(bins=15, figsize=[12, 6])
plt.show()

In [ ]:
%cd surge-sense
from rate_deviation import rate_deviation
from plot_rate_deviation import plot_diverging_plot, plot_confidence_heatmap

In [ ]:
data.head()

In [ ]:
result = rate_deviation(
    data=data,
    feature='day_of_week',
    outcome='department_referral',
    exclude_value='No referral.'
)
result.tidy

In [ ]:
result.summary()

In [ ]:
fig1 = plot_diverging_plot(result=result, feature='day_of_week', outcome=
                           'department_referral')
fig2 = plot_confidence_heatmap(result, feature='day_of_week', outcome=
                               'department_referral')
plt.show()

In [ ]:
from pathlib import Path
features = [
    'patient_race',
    'patient_gender',
    'age_category',
    'time_category',
    'day_of_week'
]

outcome = 'department_referral'
plot_dir = Path('rate_deviation_plots')
plot_dir.mkdir(exist_ok=True)

results = {}
ranking_rows = []


for feature in features:
    result = rate_deviation(
        data=data,
        feature=feature,
        outcome=outcome,
        exclude_value='No referral.',
    )
    results[feature] = result
    
    fig = plot_diverging_plot(
        result=result,
        feature=feature,
        outcome=outcome,
    )
    fig.savefig(
        plot_dir / f'{feature}_diverging.png',
        dpi=200,
        bbox_inches='tight',
    )
    plt.close(fig)
    
    fig = plot_confidence_heatmap(
        result=result,
        feature=feature,
        outcome=outcome,
    )
    fig.savefig(
        plot_dir / f'{feature}_confidence_heatmap.png',
        dpi=200,
        bbox_inches='tight',
    )
    plt.close(fig)
    
    ranking_rows.append({
        'feature': feature,
        'chi2_statistic': result.chi2_statistic,
        'p_value': result.chi2_pvalue,
        'degrees_of_freedom': result.dof,
        'significant_cells': 
            result.tidy['significant'].sum(),
        'tested_cells': result.n_cells,
    })
p_value_ranking = (
    pd.DataFrame(ranking_rows)
    .sort_values('p_value', ascending=False)
    .reset_index(drop=True)
)
p_value_ranking

In [ ]:
for feature in features:
    for feature in features:
        result = rate_deviation(
            data=data,
            feature=feature,
            outcome=outcome,
            exclude_value='No referral.',
        )
        results[feature] = result

        print(f"\n=== Feature: {feature} ===")
        print(result.summary())

In [ ]:
print(data['patient_race'].unique())
print(data['department_referral'].unique())

In [ ]:
data['patient_race'].value_counts(normalize=True)

In [ ]:
# this code allows me to only see the people who were referred to a department
referred = data.loc[data[
    'department_referral'] != 'No referral.', 'department_referral']

In [ ]:
referred.value_counts(normalize=True)

In [ ]:
data['patient_race'].value_counts(
    normalize=True).mul(100).round(1)

In [ ]:
# is referred but adds race for proper percentage calculations
referrals = data.loc[
    data['department_referral'].ne('No referral.'),
    ['patient_race', 'department_referral']
]

# the percentage of patients of that race referred to that department
crosstab_race_dpt = (
    pd.crosstab(
        referrals['patient_race'],
        referrals['department_referral'],
        normalize='columns' # index means to sums to 100 for the row; columns for the column
    )
    .mul(100)
)
# for index: given a patient is of a certain race, what percentage of their referrals go to each department
# for columns: given a pateint was referred to a certain department, what's the racial breakdown of that department
crosstab_race_dpt.round(1)

In [ ]:
referrals['patient_race'].value_counts(
    normalize=True).mul(100).round(1)

In [ ]:
baseline = (
    referrals['department_referral'].value_counts(
        normalize=True).
    mul(100).
    reindex(crosstab_race_dpt.columns)
)

difference_pp = crosstab_race_dpt.sub(
    baseline, axis='columns')
difference_pp.round(2)

In [ ]:
print(baseline)

In [ ]:
referrals_waittime = data.loc[
    data['department_referral'].ne('No referral.'),
    ['patient_waittime', 'department_referral']
]

In [ ]:
mean_values = referrals_waittime.groupby('department_referral')['patient_waittime'].mean()

plt.figure(figsize=(12, 6))
mean_values.plot(kind='bar', color=customcmap.colors)
plt.title('Average Waittime per Department Referral')
plt.xlabel('Department Referral')
plt.ylabel('Patient Waittime')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
print(data.dtypes)

In [ ]:
data.describe()

In [ ]:
columns = ['patient_race', 'patient_gender', 'age_category', 'day_of_week', 'time_category']
for col in columns:
    mean_values = data.groupby(col)['patient_waittime'].mean()

    plt.figure(figsize=(12, 6))
    mean_values.plot(kind='bar', color=customcmap.colors)
    plt.title(f"Average waittime per {col.capitalize()}")
    plt.xlabel(col.capitalize())
    plt.ylabel('Patient Waittime')
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
mean_values = data.groupby('patient_race')['patient_age'].mean()

plt.figure(figsize=(12, 6))
mean_values.plot(kind='bar', color=customcmap.colors)
plt.title('Average Age per Patient Race')
plt.xlabel('Patient Race')
plt.ylabel('Average Age')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()